In [1]:
import math
import os
import yaml
import torch
import json
import numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score, accuracy_score

from daart.data import DataGenerator, compute_sequence_pad
from daart.transforms import ZScore
from daart_utils.data import DataHandler
from daart.models import Segmenter, GMDGM, RSLDSM

In [2]:
datas = {
    'fly': {
        'vids': [
            '2019_06_26_fly2',
            '2019_08_14_fly1',
            '2019_08_20_fly3',
            '2019_10_14_fly2',
            '2019_10_21_fly1',
        ],
        'parts': ['avg', 'still', 'walk', 'front_groom', 'back_groom', 'abdomen-move'],
        'sizes': [2,3,4,5],
        'ds_name': 'fly-5'
    },
    'oft': {
        'vids': [
            'OFT_39',
            'OFT_41',
            'OFT_43',
            'OFT_44',
            'OFT_49',
            'OFT_50',
            'OFT_51',
            'OFT_52',
            'OFT_54',
            'OFT_58',
        ],
        'parts': ['avg', 'supported', 'unsupported', 'grooming'],
        'sizes': [4,6,8,10],
        'ds_name': 'mouse-oft-aligned'
    } ,
    'ibl': {
        'vids': [
            'churchlandlab_CSHL045_2020-02-27-001',
            'cortexlab_KS020_2020-02-06-001',
            'hoferlab_SWC_043_2020-09-15-001',
            'mrsicflogellab_SWC_052_2020-10-22-001',
            'wittenlab_ibl_witten_27_2021-01-21-001',
        ],
        'parts': ['avg', 'still', 'move', 'wheel_turn', 'groom'],
        'sizes': [2,3,4,5],
        'ds_name': 'ibl'
    }, 
    'huga': {
        'vids': [
            'sess_06',
            'sess_08',
            'sess_11',
            'sess_13',
            'sess_17',
        ],
        'parts': ['avg', 'walking', 'running', 'going_up', 'going_down', 'sitting',
                     'sitting_down', 'standing_up', 'standing', 'down_elevator', 'up_elevator'],
        'sizes': [100,250,500,1000],
        'ds_name': 'huga'
    } 
}

input_dict = {
    'markers': 'm',
    'features-posvel': 'fp',
    'features-sturman': 'fs',
    'features-sturman-posvel': 'fspv'
}

data_path = '/home/bsb2144/daart_utils/data/'

In [6]:
#ds = 'fly'
#ds = 'oft'
ds='ibl'
#ds='huga'

#input_type = 'markers'
#input_type = 'features-sturman'
input_type = 'features-posvel'

model_names = [
    'tcn',
    'rsl',
    'rsln',
    'gm',
    'gmnt'
]

save_names = [
    'tcn_{}'.format(input_dict[input_type]),
    'rsl_{}'.format(input_dict[input_type]),
    'rsln_{}'.format(input_dict[input_type]),
    'gm_{}'.format(input_dict[input_type]),
    'gmnt_{}'.format(input_dict[input_type]),
]


In [7]:
# note: for largest size v0 save states and latents
# loop over models
for mod_name, save_name in zip(model_names, save_names):
    # loop over data sizes
    sizes = datas[ds]['sizes']
    all_metrics = {}
    for size in sizes:
        # loop over versions
        size_metrics = []
        for v in range(5):
            # init model
            model_base = "/home/bsb2144/daart/results_daart/{}/multi-0/dtcn/".format(datas[ds]['ds_name'])
            if size == sizes[-1] and ds!='huga':
                model_dir = model_base + "{}-{}-good_sample-0_{}/version_{}".format(mod_name, size, input_type, v)
            else:
                model_dir = model_base + "{}-{}-good_sample-{}_{}/version_0".format(mod_name, size, v, input_type)
                
            model_file = os.path.join(model_dir, 'last_model.pt')
            arch_file = os.path.join(model_dir, 'hparams.yaml')
            with open(arch_file, 'rb') as f:
                hparams_new = yaml.safe_load(f)
            if hparams_new['model_class'] == 'segmenter':
                model_0 = Segmenter(hparams_new)
            elif hparams_new['model_class'] == 'rslds_marginal':
                model_0 = RSLDSM(hparams_new)
            elif hparams_new['model_class'] == 'gmdgm':
                model_0 = GMDGM(hparams_new)
            else:
                raise NotImplementedError('"%s" is an invalid model typr' % hparams_new['model_class'])
            model_0.load_state_dict(torch.load(
                model_file, map_location=lambda storage, loc: storage))

            model_0.to('cuda')
            model_0.eval()


            # loop over vids
            v_metrics = {
                'gt': [],
                'preds': []
            }
            for expt_id in datas[ds]['vids']:
                print(expt_id)
                # initialize data handler; point to correct base path
                handler = DataHandler(expt_id, base_path=os.path.join(data_path, datas[ds]['ds_name']))
                if input_type == 'markers':
                    markers_file = handler.get_marker_filepath()
                else:
                    markers_file = handler.get_feature_filepath(dirname=input_type)

                hand_labels_file = os.path.join(
                            "/home/bsb2144/daart/data/", datas[ds]['ds_name'], 'labels-hand', expt_id + '_labels.csv')

                # define data generator signals
                signals = ['markers', 'labels_strong']
                transforms = [ZScore(), None]
                paths = [markers_file, hand_labels_file]

                # build data generator
                data_gen_test = DataGenerator(
                    [expt_id], [signals], [transforms], [paths], device='cuda',#hparams['device'], 
                    batch_size=hparams_new['batch_size'], trial_splits='1;1;0;0', 
                    sequence_pad=hparams_new['sequence_pad'], sequence_length=hparams_new['sequence_length'],
                    input_type=hparams_new['input_type'])

                # load hand labels
                handler.load_hand_labels()
                states = np.argmax(handler.hand_labels.vals, axis=1)

                # compute predictions
                print('computing predictions for model 0...', end='')
                tmp = model_0.predict_labels(data_gen_test, return_scores=True)
                
                if 'tcn' in mod_name:
                    labels_pred = np.vstack(tmp['labels'][0])
                    lats = np.vstack(tmp['embedding'][0])
                elif 'rsl' in mod_name:
                    labels_pred = np.vstack(tmp['qy_x_probs'][0])
                    lats = np.vstack(tmp['qz_xy_mean'][0])
                else:
                    labels_pred = np.vstack(tmp['qy_x_probs'][0])
                    
                labels_model = np.argmax(labels_pred, axis=1)
                states = states[:len(labels_model)]
                
                # if largest size and v0 save preds and lats
                if 'tcn' in mod_name or 'rsl' in mod_name:
                    if size == sizes[-1] and v==0:
                        state_path = "/home/bsb2144/daart/metrics/{}/{}_{}_y_hat.npy".format(ds, expt_id, save_name)
                        lat_path = "/home/bsb2144/daart/metrics/{}/{}_{}_z_hat.npy".format(ds, expt_id, save_name)
                        np.save(state_path, labels_model)
                        np.save(lat_path, lats)
                
                v_metrics['gt'] += list(states)
                v_metrics['preds'] += list(labels_model)
                
            # agg results over all vids (one version)
            all_gt = np.array(v_metrics['gt'])
            all_preds = np.array(v_metrics['preds'])
            f1_by_class = f1_score(all_preds[all_gt>0], all_gt[all_gt>0], average=None)
            print('f1_by_class', f1_by_class)
            f1_avg = np.mean(f1_by_class)
            temp_res = {'avg': f1_avg}
            for part, score in zip(datas[ds]['parts'][1:], f1_by_class):
                temp_res[part] = score
            size_metrics.append(temp_res)
            print('temp_res',temp_res)
            
        # agg results over all versions (one size)
        size_temp = {}
        for part in datas[ds]['parts']:
            size_temp[part] = {}
            mean_temp = np.mean([arr[part] for arr in size_metrics])
            sd_temp = np.std([arr[part] for arr in size_metrics])/len(sizes)
            size_temp[part]['mean'] = mean_temp
            size_temp[part]['sd'] = sd_temp
            
        # save size metrics to res dict
        all_metrics[size] = size_temp
            
    # save results
    print(all_metrics)
    save_path = '/home/bsb2144/daart/metrics/{}/{}.json'.format(ds, save_name)
    with open(save_path, 'w') as f:
        json.dump(all_metrics, f)
        print('saved file to {}'.format(f))
    
    
    
    

churchlandlab_CSHL045_2020-02-27-001
NZ:  0
computing predictions for model 0...cortexlab_KS020_2020-02-06-001


/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


NZ:  0
computing predictions for model 0...hoferlab_SWC_043_2020-09-15-001
NZ:  0
computing predictions for model 0...mrsicflogellab_SWC_052_2020-10-22-001
NZ:  0
computing predictions for model 0...wittenlab_ibl_witten_27_2021-01-21-001
NZ:  0
computing predictions for model 0...f1_by_class [0.96951551 0.77010724 0.87143206 0.9372535 ]
temp_res {'avg': 0.8870770763768552, 'still': 0.9695155144256941, 'move': 0.770107238605898, 'wheel_turn': 0.8714320565991706, 'groom': 0.9372534958766583}
churchlandlab_CSHL045_2020-02-27-001
NZ:  0
computing predictions for model 0...cortexlab_KS020_2020-02-06-001


/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


NZ:  0
computing predictions for model 0...hoferlab_SWC_043_2020-09-15-001
NZ:  0
computing predictions for model 0...mrsicflogellab_SWC_052_2020-10-22-001
NZ:  0
computing predictions for model 0...wittenlab_ibl_witten_27_2021-01-21-001
NZ:  0
computing predictions for model 0...f1_by_class [0.9652572  0.89657854 0.92850511 0.99356587]
temp_res {'avg': 0.9459766786298632, 'still': 0.9652572044169135, 'move': 0.8965785381026439, 'wheel_turn': 0.9285051067780873, 'groom': 0.9935658652218082}
churchlandlab_CSHL045_2020-02-27-001
NZ:  0
computing predictions for model 0...cortexlab_KS020_2020-02-06-001


...

/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


cortexlab_KS020_2020-02-06-001
NZ:  0
computing predictions for model 0...hoferlab_SWC_043_2020-09-15-001
NZ:  0
computing predictions for model 0...mrsicflogellab_SWC_052_2020-10-22-001
NZ:  0
computing predictions for model 0...wittenlab_ibl_witten_27_2021-01-21-001
NZ:  0
computing predictions for model 0...f1_by_class [0.97655398 0.86140415 0.90007121 0.98353909]
temp_res {'avg': 0.9303921075371787, 'still': 0.9765539803707742, 'move': 0.8614041469625319, 'wheel_turn': 0.9000712081652029, 'groom': 0.9835390946502057}
churchlandlab_CSHL045_2020-02-27-001
NZ:  0
computing predictions for model 0...

/home/bsb2144/miniconda3/envs/daart2/lib/python3.7/site-packages/torch/nn/functional.py:1338: UserWarning: dropout2d: Received a 3D input to dropout2d and assuming that channel-wise 1D dropout behavior is desired - input is interpreted as shape (N, C, L), where C is the channel dim. This behavior will change in a future release to interpret the input as one without a batch dimension, i.e. shape (C, H, W). To maintain the 1D channel-wise dropout behavior, please switch to using dropout1d instead.
  warnings.warn("dropout2d: Received a 3D input to dropout2d and assuming that channel-wise "


cortexlab_KS020_2020-02-06-001
NZ:  0
computing predictions for model 0...hoferlab_SWC_043_2020-09-15-001
NZ:  0
computing predictions for model 0...mrsicflogellab_SWC_052_2020-10-22-001
NZ:  0
computing predictions for model 0...wittenlab_ibl_witten_27_2021-01-21-001
NZ:  0
computing predictions for model 0...f1_by_class [0.97426172 0.81991525 0.87670565 0.98458376]
temp_res {'avg': 0.9138665966281481, 'still': 0.9742617176916825, 'move': 0.8199152542372883, 'wheel_turn': 0.8767056530214424, 'groom': 0.9845837615621789}
{2: {'avg': {'mean': 0.8518309732607781, 'sd': 0.010598862617004238}, 'still': {'mean': 0.9235402677929475, 'sd': 0.006247037880318262}, 'move': {'mean': 0.7123734940506105, 'sd': 0.018531321144677336}, 'wheel_turn': {'mean': 0.8084518196726123, 'sd': 0.01551351322312399}, 'groom': {'mean': 0.9629583115269419, 'sd': 0.006593114164169247}}, 3: {'avg': {'mean': 0.8975271443214243, 'sd': 0.006087649338680957}, 'still': {'mean': 0.9546261200976845, 'sd': 0.0037882563307850

computing predictions for model 0...hoferlab_SWC_043_2020-09-15-001
NZ:  0
computing predictions for model 0...mrsicflogellab_SWC_052_2020-10-22-001
NZ:  0
computing predictions for model 0...wittenlab_ibl_witten_27_2021-01-21-001
NZ:  0
computing predictions for model 0...f1_by_class [0.92299847 0.75405113 0.87323245 0.94744318]
temp_res {'avg': 0.8744313087070029, 'still': 0.9229984701682815, 'move': 0.754051134317609, 'wheel_turn': 0.8732324485239394, 'groom': 0.9474431818181818}
churchlandlab_CSHL045_2020-02-27-001
NZ:  0
computing predictions for model 0...cortexlab_KS020_2020-02-06-001
NZ:  0
computing predictions for model 0...hoferlab_SWC_043_2020-09-15-001
NZ:  0
computing predictions for model 0...mrsicflogellab_SWC_052_2020-10-22-001
NZ:  0
computing predictions for model 0...wittenlab_ibl_witten_27_2021-01-21-001
NZ:  0
computing predictions for model 0...f1_by_class [0.92437404 0.7880575  0.87258304 0.97261698]
temp_res {'avg': 0.8894078928373439, 'still': 0.92437404190086

In [8]:
print('done')

done
